In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/juvecom


In [ ]:
import random
from datetime import datetime
from zoneinfo import ZoneInfo

from faker import Faker
from models.contato import Contato
from database.conexao import db, conectar

fake = Faker("pt_BR")
fuso = ZoneInfo("America/Sao_Paulo")
mensagens = [
    "Gostaria de conhecer os planos de gestão.",
    "Preciso melhorar os anúncios da minha loja.",
    "Tenho interesse na gestão de Shopee Ads.",
    "Quero ajuda para organizar minha loja.",
    "Gostaria de conversar sobre o plano Diamante.",
]

with conectar():
    with db.atomic():
        for indice in range(30):
            dia = 25 + indice % 6
            cadastro = datetime(
                2026, 9, dia,
                random.randint(8, 18),
                random.randint(0, 59),
                tzinfo=fuso
            )

            Contato.create(
                nm_contato=fake.name(),
                celular=f"11 9{fake.numerify('####-####')}",
                email=fake.ascii_safe_email(),
                mensagem=random.choice(mensagens),
                cd_status=1,
                dt_cadastro=cadastro,
                dt_atualizacao=cadastro
            )

print("30 contatos incluídos: 5 registros por dia, de 25 a 30/09/2026.")

In [ ]:

from database.conexao import conectar
from models.contato import Contato

#contato = {'nm_contato':'jh alves', 'celular':'11 1756-55632', 'email':'jh@gmail.com', 'mensagem':'este é um texto de texte'}
#contato = {'nm_contato':'banana alves', 'celular':'12 333256-55443', 'fruta':'banana' }
contato = {'nm_contato':'Novo', 'celular':'11 98865-9653', 'fruta':'banana', 'marketplace':True}


dados = Contato.incluir(contato)

print(dados)



In [ ]:

from models.contato import Contato

contatos = Contato.buscar(3)

# print(contatos)

for contato in contatos:
    print(contato.__data__)

In [ ]:

from database.conexao import conectar
from models.contato import Contato

contatos = Contato.buscar(cd_status=3)
contatos = Contato.buscar(cd_contato=3)

#print(contatos)
if contatos:
    for contato in contatos:
        print(contato.__data__)    

In [ ]:

from database.conexao import conectar
from models.contato import Contato

contatos = Contato.pendentes()

#print(contatos)
if contatos:
    for contato in contatos:
        print(contato.__data__)  


In [ ]:

from database.conexao import conectar
from models.contato import Contato

contato = {"cd_contato": 3, "cd_status": 3, "mensagem": "pra valer"}

contatos = Contato.atualizar(contato)

print(contatos)


In [ ]:

from database.conexao import conectar
from models.contato import Contato

contatos = Contato.excluir(8)

print(contatos)


In [3]:

from models.plano import Plano

planos = Plano.listar_ativos()

for plano in planos:
    print(plano.cd_plano, plano.cd_plano_tipo, plano.nm_plano, plano.periodicidade_dias)


1 1 Diamante 360
2 2 Safira 180
3 3 Esmeralda 90
4 4 Avulso 7


In [6]:

from models.plano import Plano

planos = Plano.listar_ativos()

novo_cd_plano = Plano.nova_versao(3, periodicidade_dias=120)
print(novo_cd_plano)


AttributeError: type object 'Plano' has no attribute 'nova_versao'

In [8]:

import importlib
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

print(Plano.__module__)
print(hasattr(Plano, "nova_versao"))


models.plano
True


In [14]:
from models.plano import Plano
from database.conexao import db, conectar

with conectar():
    with db.atomic() as transacao:
        alterados = (
            Plano.update(fl_ativo=False)
            .where(Plano.cd_plano == 3)
            .execute()
        )

        atual = Plano.get_by_id(3)

        print("Registros alterados:", alterados)
        print("Plano ativo:", atual.fl_ativo)

        transacao.rollback()

Registros alterados: 1
Plano ativo: False


In [12]:
novo_cd_plano = Plano.nova_versao(3, periodicidade_dias=120)
print(novo_cd_plano)

IntegrityError: duplicate key value violates unique constraint "uq_plano_tipo_ativo"
DETAIL:  Key (cd_plano_tipo)=(3) already exists.

In [13]:
import inspect

print(inspect.getsource(Plano.nova_versao))


@staticmethod
def nova_versao(cd_plano, **alteracoes):
    campos = {
        "nm_plano", "ds_plano", "periodicidade_dias", "valor",
        "texto_banner", "texto_periodo", "beneficios"
    }

    if not alteracoes or not set(alteracoes).issubset(campos):
        raise ValueError("Informe campos válidos para a nova versão.")

    with conectar():
        with db.atomic():
            atual = (
                Plano.select()
                .where(Plano.cd_plano == cd_plano)
                .for_update()
                .get_or_none()
            )

            if atual is None or not atual.fl_ativo:
                raise ValueError("Versão ativa do plano não encontrada.")

            dados = {
                "cd_plano_tipo": atual.cd_plano_tipo,
                "nm_plano": atual.nm_plano,
                "ds_plano": atual.ds_plano,
                "periodicidade_dias": atual.periodicidade_dias,
                "valor": atual.valor,
                "texto_banner": atual.texto_banner

In [15]:

from models.plano import Plano
from database.conexao import db, conectar

with conectar():
    with db.atomic() as transacao:
        atual = Plano.get_by_id(3)

        alterados = Plano.update(fl_ativo=False).where(Plano.cd_plano == 3).execute()
        print("Registros desativados:", alterados)

        novo = Plano.create(
            cd_plano_tipo=atual.cd_plano_tipo,
            nm_plano=atual.nm_plano,
            ds_plano=atual.ds_plano,
            periodicidade_dias=120,
            valor=atual.valor,
            texto_banner=atual.texto_banner,
            texto_periodo=atual.texto_periodo,
            beneficios=atual.beneficios,
            fl_ativo=True
        )

        print("Nova versão:", novo.cd_plano)
        transacao.rollback()

Registros desativados: 1
Nova versão: 8


In [16]:

novo_cd_plano = Plano.nova_versao(3, periodicidade_dias=120)
print("Nova versão:", novo_cd_plano)


IntegrityError: duplicate key value violates unique constraint "uq_plano_tipo_ativo"
DETAIL:  Key (cd_plano_tipo)=(3) already exists.

In [18]:

import importlib
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

Plano

<Model: Plano>

In [19]:
Plano.nova_versao(3, periodicidade_dias=120)

AttributeError: type object 'Plano' has no attribute 'nova_versao'

In [20]:

import importlib
import inspect
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

print("Arquivo:", inspect.getfile(Plano))
print("Método disponível:", hasattr(Plano, "nova_versao"))


Arquivo: /home/hilaneto/Jhan/Trabalho/python/projetos/juvecom/models/plano.py
Método disponível: False


In [21]:
print(inspect.getsource(Plano.nova_versao))

AttributeError: type object 'Plano' has no attribute 'nova_versao'

In [22]:
print("UPDATE - registros alterados:", alterados)
print("UPDATE - plano ativo:", Plano.get_by_id(cd_plano).fl_ativo)

UPDATE - registros alterados: 1


NameError: name 'cd_plano' is not defined

In [23]:


import inspect
import models.plano as modulo_plano

print("Métodos da classe Plano:")
print([nome for nome in dir(modulo_plano.Plano) if not nome.startswith("_")])

print("\nCódigo da classe Plano:")
print(inspect.getsource(modulo_plano.Plano))

Métodos da classe Plano:
['DoesNotExist', 'add_index', 'alias', 'beneficios', 'bind', 'bind_ctx', 'bulk_create', 'bulk_update', 'cd_plano', 'cd_plano_tipo', 'clone', 'coerce', 'copy', 'create', 'create_table', 'delete', 'delete_by_id', 'delete_instance', 'dependencies', 'dirty_field_names', 'dirty_fields', 'drop_table', 'ds_plano', 'dt_cadastro', 'filter', 'fl_ativo', 'get', 'get_by_id', 'get_id', 'get_or_create', 'get_or_none', 'index', 'insert', 'insert_from', 'insert_many', 'is_alias', 'is_dirty', 'listar_ativos', 'nm_plano', 'noop', 'periodicidade_dias', 'raw', 'replace', 'replace_many', 'save', 'select', 'set_by_id', 'table_exists', 'texto_banner', 'texto_periodo', 'truncate_table', 'unwrap', 'update', 'validate_model', 'valor']

Código da classe Plano:
class Plano(Model):
    cd_plano = BigAutoField()
    cd_plano_tipo = IntegerField()
    nm_plano = CharField(max_length=100)
    ds_plano = CharField(max_length=500, null=True)
    periodicidade_dias = IntegerField()
    valor = D

In [27]:

import importlib
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

print("Método disponível:", hasattr(Plano, "nova_versao"))

Método disponível: True


In [26]:
from pathlib import Path
import inspect
import models.plano as modulo_plano

arquivo = Path(inspect.getfile(modulo_plano.Plano))
conteudo = arquivo.read_text(encoding="utf-8")

print("Arquivo:", arquivo)
print("Método existe no arquivo:", "def nova_versao(" in conteudo)
print("Método existe na classe:", hasattr(modulo_plano.Plano, "nova_versao"))

print("\nÚltimas 80 linhas do arquivo:")
for numero, linha in list(enumerate(conteudo.splitlines(), 1))[-80:]:
    print(f"{numero:03}: {linha}")

Arquivo: /home/hilaneto/Jhan/Trabalho/python/projetos/juvecom/models/plano.py
Método existe no arquivo: True
Método existe na classe: False

Últimas 80 linhas do arquivo:
030:                 .order_by(Plano.periodicidade_dias.desc())
031:             )
032: 
033: 
034:     @staticmethod
035:     def nova_versao(cd_plano, **alteracoes):
036:         campos = {
037:             "nm_plano", "ds_plano", "periodicidade_dias", "valor",
038:             "texto_banner", "texto_periodo", "beneficios"
039:         }
040: 
041:         if not alteracoes or not set(alteracoes).issubset(campos):
042:             raise ValueError("Informe campos válidos para a nova versão.")
043: 
044:         with conectar():
045:             with db.atomic():
046:                 atual = (
047:                     Plano.select()
048:                     .where(Plano.cd_plano == cd_plano)
049:                     .for_update()
050:                     .get_or_none()
051:                 )
052: 
053:               

In [34]:

novo_cd_plano = Plano.nova_versao(3, periodicidade_dias=120)
print("Nova versão:", novo_cd_plano)

ValueError: Versão ativa do plano não encontrada.

In [33]:

import importlib
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

Plano.update(fl_ativo=False).where(Plano.cd_plano == cd_plano).execute()

novo = Plano.create(**dados)

NameError: name 'cd_plano' is not defined

In [35]:

import importlib
import models.plano as modulo_plano

importlib.reload(modulo_plano)
Plano = modulo_plano.Plano

print("Método disponível:", hasattr(Plano, "nova_versao"))
print("Planos ativos:", [(p.cd_plano, p.nm_plano) for p in Plano.listar_ativos()])

Método disponível: True
Planos ativos: [(1, 'Diamante'), (2, 'Safira'), (10, 'Esmeralda'), (4, 'Avulso')]


In [59]:
from faker import Faker
import random
import re

fake = Faker("pt_BR")


def gerar_pessoa_juridica():
    empresa = fake.company()
    cnpj = re.sub(r"\D", "", fake.cnpj())
    celular = re.sub(r"\D", "", fake.cellphone_number())
    cep = re.sub(r"\D", "", fake.postcode())

    return {
        "tp_pessoa": "J",
        "nome_fantasia": empresa,
        "cnpj": cnpj,
        "celular": celular,
        "email": fake.company_email(),
        "endereco": fake.street_name(),
        "numero": str(fake.building_number()),
        "complemento": random.choice(["", "Sala 101", "Conjunto 202", "Loja 01", "Galpão A"]),
        "bairro": fake.bairro() if hasattr(fake, "bairro") else fake.word().title(),
        "cidade": fake.city(),
        "uf": fake.estado_sigla(),
        "cep": cep,
        "nm_responsavel": fake.name()
    }


if __name__ == "__main__":
    from pprint import pprint

    pprint(gerar_pessoa_juridica(), sort_dicts=False)

{'tp_pessoa': 'J',
 'nome_fantasia': 'Costela',
 'cnpj': '15739804000184',
 'celular': '28998536075',
 'email': 'nogueiralorena@silveira.gov.br',
 'endereco': 'Rua Joana Teixeira',
 'numero': '255',
 'complemento': 'Conjunto 202',
 'bairro': 'Alpes',
 'cidade': 'Fogaça do Galho',
 'uf': 'RN',
 'cep': '57419573',
 'nm_responsavel': 'Dom Sousa'}
